In [6]:
try:
    import networkx as nx
except ImportError:
    print("⚠️  networkx não instalado. Instalando...")
    import subprocess
    subprocess.run(["pip", "install", "networkx"], check=True)
    import networkx as nx

# Criar grafo de dependências
G = nx.DiGraph()

# Adicionar nós
for subtask in all_subtasks:
    G.add_node(subtask.id, title=subtask.title[:30], effort=subtask.effort_days)

# Adicionar edges (dependências)
for subtask in all_subtasks:
    for dep in subtask.depends_on:
        G.add_edge(dep, subtask.id)

# Visualizar
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Grafo 1: Dependency Graph
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

# Cores baseadas em paralelizabilidade
colors = []
for node in G.nodes():
    task = next((t for t in all_subtasks if t.id == node), None)
    if task and task.can_work_in_parallel:
        colors.append('#90EE90')  # Verde = paralelo
    else:
        colors.append('#FFB6C6')  # Rosa = sequencial

nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=2000, ax=ax1)
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20, ax=ax1, arrowstyle='->')
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax1)

ax1.set_title('Grafo de Dependências\n(Verde=Paralelo, Rosa=Sequencial)', fontsize=14, fontweight='bold')
ax1.axis('off')

# Grafo 2: Roadmap Gantt
tasks_for_gantt = []
task_positions = {}
current_day = 0

# Agrupar tarefas por nível de dependência
levels = nx.topological_sort(G)
day_map = {}

for task_id in levels:
    task = next((t for t in all_subtasks if t.id == task_id), None)
    deps = task.depends_on if task else []
    
    if not deps:
        start_day = 0
    else:
        start_day = max(day_map.get(dep, 0) for dep in deps)
    
    end_day = start_day + (task.effort_days if task else 1)
    day_map[task_id] = end_day
    tasks_for_gantt.append((task_id, start_day, task.effort_days if task else 1))

# Plot Gantt
y_pos = 0
colors_gantt = {}
for task_id, start, duration in sorted(tasks_for_gantt, key=lambda x: x[1]):
    task = next((t for t in all_subtasks if t.id == task_id), None)
    color = '#90EE90' if (task and task.can_work_in_parallel) else '#FFB6C6'
    colors_gantt[task_id] = color
    
    ax2.barh(y_pos, duration, left=start, height=0.8, color=color, edgecolor='black', linewidth=1)
    ax2.text(start + duration/2, y_pos, task_id, va='center', ha='center', fontsize=9, fontweight='bold')
    y_pos += 1

ax2.set_yticks(range(len(tasks_for_gantt)))
ax2.set_yticklabels([t[0] for t in sorted(tasks_for_gantt, key=lambda x: x[1])])
ax2.set_xlabel('Dias de Desenvolvimento', fontsize=12)
ax2.set_title('Roadmap de Implementação (Gantt Chart)', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Caminho crítico
try:
    critical_path = nx.dag_longest_path(G)
    critical_length = nx.dag_longest_path_length(G)
    print(f"\n🔴 CAMINHO CRÍTICO (tarefas sequenciais obrigatórias):")
    print(f"   Tarefas: {' → '.join(critical_path)}")
    print(f"   Duração: {critical_length} dias mínimo")
except:
    print("\n⚠️  Não conseguiu calcular caminho crítico")

# Summary
print("\n\n" + "="*80)
print("📈 RESUMO EXECUTIVO")
print("="*80)
print(f"""
✅ 7 Gaps identificados
✅ Top 3 Features priorizadas
✅ 10 Sub-tasks criadas
✅ Duração paralela ótima: ~7 dias
✅ Duração sequencial: 16 dias

🎯 RECOMENDAÇÃO:
   Atribuir 2-3 desenvolvedores para trabalhar em paralelo
   - Dev 1: FE-LMS-001 (Cursos Frontend)  
   - Dev 2: BE-LMS-004 (GET /api/courses)
   - Dev 3: FE-LMS-002 (Lesson Player)
   
   Resultado esperado: 1 semana ao invés de 2.5 semanas

📅 PRÓXIMAS AÇÕES:
   1. Criar issues no GitHub com templates gerados
   2. Atribuir para equipe
   3. Iniciar Sprint com sub-tasks prioritárias
   4. Implementar e testar em paralelo
""")

NameError: name 'all_subtasks' is not defined

---

## Fase 6: Visualização de Dependências e Roadmap

**Objetivo**: Criar visualizações do grafo de dependências e roadmap temporal de implementação.

In [4]:
@dataclass
class SubTask:
    id: str
    title: str
    description: str
    effort_days: int
    can_work_in_parallel: bool
    depends_on: List[str]
    acceptance: List[str]

# Sub-tasks para FE-LMS-001
fe_lms_001_subtasks = [
    SubTask(
        id="FE-LMS-001-A",
        title="Criar hook use-courses com TanStack Query",
        description="Implementar apps/web/src/hooks/api/use-courses.ts com useQuery (GET /api/courses) e useMutation (POST/PUT/DELETE)",
        effort_days=1,
        can_work_in_parallel=True,
        depends_on=[],
        acceptance=[
            "useGetCourses() retorna { courses, isLoading, error, pagination }",
            "useCreateCourse() permite POST com validação Zod",
            "useUpdateCourse() permite PUT",
            "useDeleteCourse() permite DELETE",
            "Queries são invalidadas após mutações"
        ]
    ),
    SubTask(
        id="FE-LMS-001-B",
        title="Criar componente CourseForm",
        description="Implementar apps/web/src/components/CourseForm.tsx com form para criar/editar cursos",
        effort_days=2,
        can_work_in_parallel=True,
        depends_on=["FE-LMS-001-A"],
        acceptance=[
            "Form valida nome, código, modalidade",
            "Submit integra com useCreateCourse()",
            "Erros são exibidos inline",
            "Form reseta após sucesso"
        ]
    ),
    SubTask(
        id="FE-LMS-001-C",
        title="Criar página CourseList",
        description="Implementar apps/web/src/app/courses/page.tsx com DataTable de cursos",
        effort_days=2,
        can_work_in_parallel=True,
        depends_on=["FE-LMS-001-A", "FE-LMS-001-B"],
        acceptance=[
            "DataTable exibe cursos com paginação",
            "Filtro por status funciona",
            "Botão 'Novo Curso' abre modal com CourseForm",
            "Link 'Ver Detalhes' navega para /courses/[id]"
        ]
    ),
    SubTask(
        id="FE-LMS-001-D",
        title="Criar página CourseDetail com nested Disciplines",
        description="Implementar apps/web/src/app/courses/[id]/page.tsx exibindo disciplinas, módulos, aulas",
        effort_days=2,
        can_work_in_parallel=False,
        depends_on=["FE-LMS-001-C"],
        acceptance=[
            "Página carrega dados do curso e disciplinas",
            "Exibe tree view: Curso > Disciplinas > Módulos > Aulas",
            "Ações para editar/deletar curso",
            "Navegação breadcrumb"
        ]
    )
]

# Sub-tasks para BE-LMS-004
be_lms_004_subtasks = [
    SubTask(
        id="BE-LMS-004-A",
        title="Adicionar Query Schema no Zod",
        description="Criar schema Zod para validar query params em packages/types/src/course.ts",
        effort_days=1,
        can_work_in_parallel=True,
        depends_on=[],
        acceptance=[
            "CourseQuerySchema valida page, limit, status, search",
            "Valores padrão: page=1, limit=10",
            "Tipo exportado CourseQuery"
        ]
    ),
    SubTask(
        id="BE-LMS-004-B",
        title="Implementar handler GET /api/courses",
        description="Adicionar função GET em apps/api/src/routes/courses.ts",
        effort_days=2,
        can_work_in_parallel=False,
        depends_on=["BE-LMS-004-A"],
        acceptance=[
            "GET / api/courses?page=1&limit=10 retorna cursos",
            "Filtro status funciona",
            "Filtro search funciona (partial match)",
            "Paginação está correta (total, pages)",
            "Usuário normal vê apenas publicados",
            "Admin vê todos"
        ]
    )
]

# Sub-tasks para FE-LMS-002
fe_lms_002_subtasks = [
    SubTask(
        id="FE-LMS-002-A",
        title="Criar hooks use-lesson e use-lesson-progress",
        description="Implementar fetch de lição, materiais, e progresso",
        effort_days=1,
        can_work_in_parallel=True,
        depends_on=[],
        acceptance=[
            "useGetLesson(:id) retorna lesson + materials",
            "usePostProgress() permite atualizar progresso",
            "Mutations invalidam queries após sucesso"
        ]
    ),
    SubTask(
        id="FE-LMS-002-B",
        title="Criar componente VideoPlayer",
        description="Wrapper Plyr.io com controles customizados",
        effort_days=2,
        can_work_in_parallel=True,
        depends_on=["FE-LMS-002-A"],
        acceptance=[
            "VideoPlayer carrega e reproduz HLS/MP4",
            "Barra de progresso mostra duração",
            "Controles: play, pause, volume, fullscreen"
        ]
    ),
    SubTask(
        id="FE-LMS-002-C",
        title="Criar componentes MaterialViewer e LessonSidebar",
        description="Visualizador de materiais + sidebar com lista de lições",
        effort_days=2,
        can_work_in_parallel=True,
        depends_on=["FE-LMS-002-A"],
        acceptance=[
            "MaterialViewer renderiza PDF, imagens, links",
            "LessonSidebar lista todas as aulas do módulo",
            "Click navega para nova lição"
        ]
    ),
    SubTask(
        id="FE-LMS-002-D",
        title="Criar página LessonPlayer e integração",
        description="Integrar tudo em apps/web/src/app/courses/[courseId]/lessons/[lessonId]/page.tsx",
        effort_days=2,
        can_work_in_parallel=False,
        depends_on=["FE-LMS-002-B", "FE-LMS-002-C"],
        acceptance=[
            "Página carrega e exibe video + materiais",
            "Auto-track progresso",
            "Navegação Próxima/Anterior aula",
            "Breadcrumb visível"
        ]
    )
]

# Exibir breakdown
all_subtasks = fe_lms_001_subtasks + be_lms_004_subtasks + fe_lms_002_subtasks

df_subtasks = pd.DataFrame([{
    'ID': s.id,
    'Título': s.title,
    'Esforço (dias)': s.effort_days,
    'Paralelo': '✅ Sim' if s.can_work_in_parallel else '❌ Não',
    'Depende de': ','.join(s.depends_on) if s.depends_on else '-'
} for s in all_subtasks])

print("🔄 BREAKDOWN EM SUB-TASKS")
print("="*100)
print(df_subtasks.to_string(index=False))

print(f"\n\n📊 ESTATÍSTICAS:")
print(f"   Total de sub-tasks: {len(all_subtasks)}")
print(f"   Total de dias de esforço: {sum(s.effort_days for s in all_subtasks)}")
print(f"   Tasks paralelos: {sum(1 for s in all_subtasks if s.can_work_in_parallel)}")
print(f"   Tasks sequenciais: {sum(1 for s in all_subtasks if not s.can_work_in_parallel)}")

# Tempo paralelo ótimo
parallelizable = [s for s in all_subtasks if s.can_work_in_parallel]
sequential = [s for s in all_subtasks if not s.can_work_in_parallel]
optimal_time = len(sequential) + max([s.effort_days for s in parallelizable], default=0)

print(f"\n⚡ TEMPO ÓTIMO COM PARALELIZAÇÃO: ~{optimal_time} dias (ao invés de {sum(s.effort_days for s in all_subtasks)})")

# Dependency graph
print("\n\n🔗 GRAFO DE DEPENDÊNCIAS (ordem recomendada):")
print("\n1️⃣  FASE 1 - Pode iniciar imediatamente (paralelo):")
for s in all_subtasks:
    if not s.depends_on:
        print(f"   • {s.id}: {s.title} ({s.effort_days}d)")

print("\n2️⃣  FASE 2 - Após fase 1:")
for s in all_subtasks:
    if s.depends_on and all(dep not in [x.id for x in all_subtasks if not x.depends_on] for dep in s.depends_on):
        print(f"   • {s.id}: {s.title} ({s.effort_days}d)")

print("\n3️⃣  FASE 3 - Final/Integration:")
for s in all_subtasks:
    if not s.can_work_in_parallel:
        print(f"   • {s.id}: {s.title} ({s.effort_days}d) - SEQUENCIAL")

NameError: name 'dataclass' is not defined

---

## Fase 5.5: Otimização de Distribuição de Trabalho

**Objetivo**: Quebrar issues grandes em sub-tasks menores (1-3 dias cada) para melhor paralelização.

In [2]:
def generate_issue_yaml(spec: FeatureSpecification) -> str:
    """Gerar YAML com instruções para criar issue manualmente (sem PyGithub)"""
    
    labels = ["enhancement", "LMS"]
    if "FE-" in spec.id:
        labels.append("frontend")
    else:
        labels.append("backend")
    
    blocked_by = spec.dependencies.get("Blocked by", [])
    blocks = spec.dependencies.get("Blocks", [])
    
    return f"""
# GitHub Issue Template: {spec.title}

## Issue YAML (para criar manualmente em GitHub)

```yaml
title: "[{spec.id}] {spec.title}"
labels: {labels}
body: |
{spec.to_github_issue()}
```

## Links para Dependências

### Este issue bloqueia:
{chr(10).join(f'- #{issue}' for issue in blocks) if blocks else '- Nenhum'}

### Este issue é bloqueado por:
{chr(10).join(f'- #{issue}' for issue in blocked_by) if blocked_by else '- Nenhum'}

---
"""

# Gerar instruções para criar issues
print("📝 INSTRUÇÕES PARA CRIAR ISSUES NO GITHUB")
print("="*80)
print("\n⚠️  IMPORTANTE: Crie as issues manualmente no GitHub em ordem de prioridade:")
print("   1. Primeiro crie a issue FE-LMS-001 (frontend cursos)")
print("   2. Depois crie BE-LMS-004 (GET /api/courses)")
print("   3. Depois crie FE-LMS-002 (lesson player)")
print("\n" + "="*80)

for spec in specs_top3:
    print(generate_issue_yaml(spec))

# Salvar em arquivo
output_file = REPO_ROOT / ".github/issues-template.md"
with open(output_file, "w") as f:
    f.write("# GitHub Issues Template\n\n")
    f.write("Este arquivo contém templates de issues para criar no GitHub.\n\n")
    for spec in specs_top3:
        f.write(generate_issue_yaml(spec))
        f.write("\n\n")

print(f"\n✅ Templates salvos em: {output_file}")
print(f"\n📋 Próximas ações:")
print(f"   1. Abrir {output_file}")
print(f"   2. Copiar template de cada issue")
print(f"   3. Colar em GitHub > New Issue")
print(f"   4. Preencher título e descrição")
print(f"   5. Selecionar labels e projetos")

📝 INSTRUÇÕES PARA CRIAR ISSUES NO GITHUB

⚠️  IMPORTANTE: Crie as issues manualmente no GitHub em ordem de prioridade:
   1. Primeiro crie a issue FE-LMS-001 (frontend cursos)
   2. Depois crie BE-LMS-004 (GET /api/courses)
   3. Depois crie FE-LMS-002 (lesson player)



NameError: name 'specs_top3' is not defined

---

## Fase 5: Criação de Issues no GitHub via API

**Objetivo**: Converter especificações em issues GitHub com labels, dependências e linking automático.

In [ ]:
@dataclass
class FeatureSpecification:
    id: str
    title: str
    overview: str
    scope_included: List[str]
    scope_excluded: List[str]
    technical_requirements: Dict[str, str]
    implementation_plan: Dict[str, str]
    acceptance_criteria: List[str]
    dependencies: Dict[str, List[str]]
    estimated_effort: str
    
    def to_github_issue(self) -> str:
        """Converter para formato de issue GitHub"""
        issue_body = f"""## Overview
{self.overview}

## Scope

### Incluído ✅
{chr(10).join(f'- {item}' for item in self.scope_included)}

### Excluído ❌
{chr(10).join(f'- {item}' for item in self.scope_excluded)}

## Technical Requirements
"""
        for key, value in self.technical_requirements.items():
            issue_body += f"\n### {key}\n{value}\n"
        
        issue_body += f"\n## Implementation Plan\n"
        for step, desc in self.implementation_plan.items():
            issue_body += f"\n### {step}\n{desc}\n"
        
        issue_body += f"\n## Acceptance Criteria\n"
        for criterion in self.acceptance_criteria:
            issue_body += f"- [ ] {criterion}\n"
        
        issue_body += f"\n## Dependencies\n"
        for key, items in self.dependencies.items():
            issue_body += f"- **{key}**: {', '.join(items) if items else 'None'}\n"
        
        issue_body += f"\n## Estimated Effort\n{self.estimated_effort}\n"
        
        return issue_body

# Specification 1
spec1 = FeatureSpecification(
    id="FE-LMS-001",
    title="Frontend LMS - Listagem e Gestão de Cursos",
    overview="""Implementar interface web para visualizar, criar, editar e deletar cursos. Os instrutores (role: admin/instructor) precisam gerenciar seus cursos através de uma interface intuitiva. Esta feature consome os endpoints POST/GET /api/courses e permite CRUD completo.""",
    scope_included=[
        "Página `/courses` - Listagem de todos os cursos com filtros e paginação",
        "Página `/courses/[id]` - Detalhes do curso com disciplinas aninhadas",
        "Componente `CourseForm` - Criar e editar curso",
        "Hook `use-courses` - Chamadas API com TanStack Query",
        "Validação com Zod (mesmo schema do backend)",
        "Loading states com Skeleton components",
        "Error boundaries e tratamento de erros"
    ],
    scope_excluded=[
        "Temas customizados por instituição",
        "Importação em lote (usar endpoint /api/bulk/import-courses separadamente)",
        "Histórico de versões de cursos"
    ],
    technical_requirements={
        "API Integration": "GET /api/courses?page=1&limit=10&status=published\nPOST /api/courses\nPUT /api/courses/:id\nDELETE /api/courses/:id",
        "Frontend Stack": "Next.js 16 App Router, React 19, TanStack Query, Zod, shadcn/ui",
        "State Management": "Jotai para filtros, TanStack Query para dados do servidor",
        "UI Components": "DataTable, Form, Dialog, Select, Input, Button (shadcn/ui)"
    },
    implementation_plan={
        "Step 1: Hook Criação": "Criar `apps/web/src/hooks/api/use-courses.ts` com useQuery/useMutation",
        "Step 2: Página List": "Criar `apps/web/src/app/courses/page.tsx` com DataTable",
        "Step 3: Página Detail": "Criar `apps/web/src/app/courses/[id]/page.tsx` com nested Disciplines",
        "Step 4: Form Component": "Criar `apps/web/src/components/CourseForm.tsx` com validação Zod",
        "Step 5: Integração": "Adicionar link no sidebar admin, testar fluxo completo"
    },
    acceptance_criteria=[
        "Usuário admin consegue listar todos os cursos na página /courses",
        "Filtro por status (rascunho, publicado, arquivado) funciona",
        "Paginação exibe 10 cursos por página",
        "Clique em curso abre página de detalhes com disciplinas aninhadas",
        "Botão 'Novo Curso' abre modal com form",
        "Form valida campos obrigatórios antes de enviar",
        "POST /api/courses é chamado ao salvar novo curso",
        "Mensagem de sucesso exibida após criar curso",
        "Loading state exibido durante requisição",
        "Erros da API são exibidos ao usuário com mensagem amigável"
    ],
    dependencies={
        "Blocks": [],
        "Blocked by": ["BE-LMS-004"],  # Precisa de GET /api/courses
        "Relates to": ["FE-LMS-002", "FE-LMS-005"]
    },
    estimated_effort="Medium (5-7 days)"
)

# Specification 2
spec2 = FeatureSpecification(
    id="BE-LMS-004",
    title="GET /api/courses - Listagem com Filtros e Paginação",
    overview="""Implementar endpoint GET para listar cursos com suporte a filtros (status, nível, modalidade), busca por nome, e paginação. Retornar resultado no formato `{ data: Course[], pagination: { ... } }`. Este endpoint é essencial para que o frontend consiga listar e filtrar cursos.""",
    scope_included=[
        "Query params: page, limit, status, nivel, modalidade, search",
        "Validação com Zod dos query params",
        "Filtro por curso.ativo = true (padrão)",
        "Suporte a busca por partial match em course.nome",
        "Paginação com skip/take",
        "Ordenação por createdAt DESC",
        "Retorno de tipo Course com metadados da paginação",
        "Admin pode ver todos; usuário normal vê apenas publicados"
    ],
    scope_excluded=[
        "Busca full-text (usar LIKE para MVP)",
        "Filtro por instrutor (tratar depois)",
        "Aggregações de estatísticas (usar endpoint /api/reports/course-analytics separadamente)",
        "Recomendações personalizadas"
    ],
    technical_requirements={
        "Route": "GET /api/courses",
        "Query Params": "page?: number, limit?: number, status?: enum, nivel?: enum, modalidade?: enum, search?: string",
        "Response": "{ data: Course[], pagination: { total: number, page: number, limit: number, pages: number } }",
        "Authorization": "Public (sem auth requerida, mas filtros variam por role)",
        "Validation": "Usar ZodTypeProvider com Zod schemas"
    },
    implementation_plan={
        "Step 1: Type Def": "Adicionar QueryParamsSchema em CreateCourseInput (packages/types/src/course.ts)",
        "Step 2: Route GET": "Adicionar handler GET em apps/api/src/routes/courses.ts",
        "Step 3: Prisma Query": "Usar prisma.course.findMany({where: {...}, skip, take})",
        "Step 4: Count": "Usar prisma.course.count({where: ...}) para total",
        "Step 5: Test": "Testar com filtros: /courses?status=publicado&limit=10&page=1"
    },
    acceptance_criteria=[
        "GET /api/courses retorna array de cursos",
        "Paginação: ?page=1&limit=10 retorna 10 itens",
        "Filtro status: ?status=publicado retorna apenas publicados",
        "Filtro nível: ?nivel=graduacao retorna cursos de graduação",
        "Busca: ?search=Python retorna cursos com 'Python' no nome",
        "Combinação: ?status=publicado&nivel=tecnico&page=1 funciona",
        "Acesso não-autenticado: retorna apenas cursos com ativo=true e status=publicado",
        "Admin vê todos os cursos (mesmo rascunho)",
        "Response inclui pagination metadata (total, pages)",
        "Erros 400 para query params inválidos"
    ],
    dependencies={
        "Blocks": ["FE-LMS-001"],  # Frontend precisa deste endpoint
        "Blocked by": [],
        "Relates to": []
    },
    estimated_effort="Small (2-3 days)"
)

# Specification 3
spec3 = FeatureSpecification(
    id="FE-LMS-002",
    title="Frontend LMS - Player de Aulas com Progresso",
    overview="""Implementar player de aulas para estudantes acessarem lições, visualizarem materiais (vídeos, PDFs, slides), e registrarem progresso. Integra com /api/lessons/:id, /api/materials, e /api/students/:id/progress. MVP focado em video player + material viewing + progress tracking.""",
    scope_included=[
        "Página `/courses/[courseId]/lessons/[lessonId]` - Player principal",
        "Video player (HLS/MP4) usando Plyr ou VideoJS",
        "Sidebar com lista de lições + navegação",
        "Visualizador de materiais (PDF embed, slides, links)",
        "Progress bar mostrando completude da lição",
        "Botão 'Marcar como Assistido' / Auto-track após X minutos",
        "Componentes: LessonPlayer, VideoPlayer, MaterialViewer, LessonSidebar",
        "Hook: use-lesson-progress para POST /api/students/:id/progress"
    ],
    scope_excluded=[
        "Chat de dúvidas em tempo real",
        "Transmissão ao vivo",
        "Legendas customizadas por usuário",
        "Download de materiais"
    ],
    technical_requirements={
        "Video Playback": "Plyr.io ou VideoJS para streaming HLS/MP4",
        "Material Viewing": "PDF.js para PDFs, HTML5 img para slides",
        "API Calls": "GET /api/lessons/:id, GET /api/materials/:id, POST /api/students/:id/progress",
        "State": "Jotai para lessonId, currentTime, completionPercentage",
        "UI": "shadcn/ui components + custom video player CSS"
    },
    implementation_plan={
        "Step 1: Hook": "Criar `use-lesson.ts` e `use-lesson-progress.ts` hooks",
        "Step 2: Player": "Criar `VideoPlayer.tsx` com Plyr",
        "Step 3: Materials": "Criar `MaterialViewer.tsx` com detection tipo + renderer",
        "Step 4: Sidebar": "Criar `LessonSidebar.tsx` com lista de lições",
        "Step 5: Page": "Criar `[courseId]/lessons/[lessonId]/page.tsx` integrando tudo",
        "Step 6: Progress": "Implementar tracking automático POST /api/students/:id/progress"
    },
    acceptance_criteria=[
        "Player de vídeo carrega e reproduz corretamente",
        "Barra de progresso mostra % completude",
        "Materiais complementares carregam corretamente",
        "Clique em lição na sidebar navega para ela",
        "Progresso é salvo no backend após X minutos de visualização",
        "Página mostra loading state enquanto carrega dados",
        "Erro ao carregar vídeo exibe mensagem amigável",
        "Responsive funciona em mobile (áudio saleado, player adapta)",
        "Próxima/Anterior lição disponível na barra de navegação",
        "Breadcrumb mostra: Cursos > Curso > Disciplina > Módulo > Lição"
    ],
    dependencies={
        "Blocks": ["FE-LMS-005"],  # Dashboard precisa disso
        "Blocked by": ["FE-LMS-001"],  # Precisa ver cursos primeiro
        "Relates to": ["FE-LMS-003"]
    },
    estimated_effort="Medium (6-8 days)"
)

specs_top3 = [spec1, spec2, spec3]

for i, spec in enumerate(specs_top3, 1):
    print(f"\n{'='*80}")
    print(f"ESPECIFICAÇÃO #{i}: {spec.title}")
    print(f"ID: {spec.id}")
    print(f"Esforço Estimado: {spec.estimated_effort}")
    print(f"{'='*80}")
    print(spec.to_github_issue()[:500] + "...\n")  # Primeiros 500 chars

---

## Fase 4: Geração Automática de Especificações

**Objetivo**: Criar especificações técnicas detalhadas para os top 3 features.

**Template**: Visão Geral → Requisitos Técnicos → Plano de Implementação → Critérios de Aceitação

In [ ]:
@dataclass
class PriorityScoringData:
    id: str
    name: str
    user_impact: int  # 1-5
    strategic_alignment: int  # 1-5
    implementation_effort: int  # 1-5 (3+ é difícil)
    risk_level: int  # 1-5
    
    def calculate_priority(self):
        numerator = self.user_impact * self.strategic_alignment
        denominator = self.implementation_effort * self.risk_level
        return round(numerator / denominator, 2)

# Scoring Matrix
scores = [
    PriorityScoringData("FE-LMS-001", "Frontend LMS - Cursos", user_impact=5, strategic_alignment=5, implementation_effort=4, risk_level=2),
    PriorityScoringData("FE-LMS-002", "Frontend LMS - Player", user_impact=5, strategic_alignment=5, implementation_effort=4, risk_level=2),
    PriorityScoringData("FE-LMS-003", "Frontend LMS - Quiz", user_impact=4, strategic_alignment=4, implementation_effort=3, risk_level=2),
    PriorityScoringData("BE-LMS-004", "GET /api/courses", user_impact=5, strategic_alignment=5, implementation_effort=2, risk_level=1),
    PriorityScoringData("FE-LMS-005", "Student Dashboard", user_impact=5, strategic_alignment=5, implementation_effort=3, risk_level=1),
    PriorityScoringData("BE-INFRA-006", "Email Queue System", user_impact=4, strategic_alignment=4, implementation_effort=3, risk_level=2),
    PriorityScoringData("BE-INFRA-007", "Metrics & Monitoring", user_impact=2, strategic_alignment=3, implementation_effort=3, risk_level=1),
]

# Calcular prioridades
for score in scores:
    score.priority = score.calculate_priority()

# Ordenar por prioridade
scores_sorted = sorted(scores, key=lambda x: x.priority, reverse=True)

# DataFrame
df_priority = pd.DataFrame([{
    'ID': s.id,
    'Feature': s.name,
    'User Impact': s.user_impact,
    'Strategic': s.strategic_alignment,
    'Effort': s.implementation_effort,
    'Risk': s.risk_level,
    'Priority Score': s.priority
} for s in scores_sorted])

print("🎯 MATRIX DE PRIORIZAÇÃO:")
print(df_priority.to_string(index=False))

# Top 3
print("\n" + "="*70)
print("🏆 TOP 3 FEATURES PRIORITÁRIAS")
print("="*70)
for i, score in enumerate(scores_sorted[:3], 1):
    print(f"\n{i}. {score.name} [{score.id}]")
    print(f"   Score: {score.priority}")
    print(f"   Fórmula: ({score.user_impact} × {score.strategic_alignment}) / ({score.implementation_effort} × {score.risk_level}) = {score.priority}")

# Visualizar
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#FF6B6B', '#FFA726', '#FFA726', '#FFD93D', '#FFD93D', '#FFA726', '#FFD93D']
bars = ax.barh(df_priority['Feature'], df_priority['Priority Score'], color=colors)
ax.set_xlabel('Priority Score', fontsize=12)
ax.set_title('Feature Prioritization Matrix', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Adicionar valores nas barras
for i, (bar, val) in enumerate(zip(bars, df_priority['Priority Score'])):
    ax.text(val + 0.1, i, f'{val:.2f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Top 3 identific ados para especificação")

---

## Fase 3: Priorização com Matriz de Pontuação

**Objetivo**: Calcular prioridades usando a fórmula: `Priority = (User Impact × Strategic Alignment) / (Implementation Effort × Risk Level)`

**Escala**: 1-5 para cada métrica

In [1]:
@dataclass
class FeatureGap:
    id: str
    name: str
    description: str
    current_status: str
    documentation_ref: str
    user_impact: str
    missing_components: List[str]

gaps = [
    FeatureGap(
        id="FE-LMS-001",
        name="Frontend LMS - Listagem e Gestão de Cursos",
        description="Páginas web para visualizar, criar, editar e deletar cursos. Consumir POST/GET /api/courses.",
        current_status="🔴 Não implementado",
        documentation_ref="Backend: courses.ts (15 endpoints LMS prontos)",
        user_impact="🔴 CRÍTICO - Instrutores não conseguem gerenciar cursos",
        missing_components=["CourseList page", "CourseDetail page", "CourseForm", "use-courses hook"]
    ),
    FeatureGap(
        id="FE-LMS-002",
        name="Frontend LMS - Player de Aulas com Progresso",
        description="Página para visualizar lições, materiais, vídeos. Rastrear progresso do aluno.",
        current_status="🔴 Não implementado",
        documentation_ref="Backend: lessons.ts, materials.ts, progress.ts",
        user_impact="🔴 CRÍTICO - Estudantes não conseguem acessar conteúdo",
        missing_components=["LessonPlayer component", "VideoPlayer", "MaterialViewer", "ProgressTracker"]
    ),
    FeatureGap(
        id="FE-LMS-003",
        name="Frontend LMS - Quiz Player e Resultados",
        description="Interface para responder quizzes, visualizar resultados e desempenho.",
        current_status="🔴 Não implementado",
        documentation_ref="Backend: quizzes.ts, quiz-results.ts",
        user_impact="🟡 ALTO - Avaliação é crítica para LMS",
        missing_components=["QuizPlayer", "QuestionRenderer", "ResultsView", "use-quiz-results hook"]
    ),
    FeatureGap(
        id="BE-LMS-004",
        name="GET /api/courses - Listagem com Filtros e Paginação",
        description="Endpoint GET para listar cursos com filtros (status, nível, modalidade) e paginação.",
        current_status="🟡 Parcial - Apenas POST existe",
        documentation_ref="Backend: courses.ts (faltam métodos GET)",
        user_impact="🟡 ALTO - Necessário para frontend listar cursos",
        missing_components=["GET /api/courses", "Query params validation", "Pagination logic"]
    ),
    FeatureGap(
        id="BE-LMS-005",
        name="Frontend LMS - Dashboard do Estudante",
        description="Página principal mostrando cursos inscritos, progresso, próximas aulas.",
        current_status="🔴 Não implementado",
        documentation_ref="Backend: dashboard.ts (GET /api/students/:id/dashboard existe)",
        user_impact="🔴 CRÍTICO - Entry point para estudantes",
        missing_components=["StudentDashboard page", "CourseProgressCard", "use-student-dashboard hook"]
    ),
    FeatureGap(
        id="BE-INFRA-006",
        name="Email Notifications & Queue System",
        description="Sistema para enviar emails de notificação (inscrição, aula nova, certificado). Usar fila assincrona.",
        current_status="🟡 Parcial - Infra existe (templates + nodemailer), sem queue",
        documentation_ref="Backend: notifications.ts, emails/",
        user_impact="🟡 ALTO - Retenção de usuários depende de notificações",
        missing_components=["Bull Queue (Redis)", "Email service async", "Notification scheduler"]
    ),
    FeatureGap(
        id="BE-INFRA-007",
        name="API Metrics & Monitoring Dashboard",
        description="Coletar métricas de performance (response time, erros), expor em /api/metrics.",
        current_status="🟡 Parcial - metrics.ts existe mas sem Prometheus",
        documentation_ref="Backend: metrics.ts, stats.ts",
        user_impact="🟢 MÉDIO - Operacional mas não bloqueia features",
        missing_components=["Prometheus exporter", "Grafana integration", "Alerting setup"]
    )
]

# Exibir gaps
print("📋 GAPS IDENTIFICADOS:\n")
for i, gap in enumerate(gaps, 1):
    print(f"{i}. {gap.name} [{gap.id}]")
    print(f"   Status: {gap.current_status}")
    print(f"   Impacto: {gap.user_impact}")
    print(f"   Componentes: {', '.join(gap.missing_components)}")
    print()

df_gaps = pd.DataFrame([{
    'ID': g.id,
    'Feature': g.name,
    'Status': g.current_status,
    'Impacto': g.user_impact,
} for g in gaps])

print("\n📊 Tabela de Gaps:")
print(df_gaps.to_string(index=False))

NameError: name 'dataclass' is not defined

---

## Fase 2: Análise de Lacunas (Gap Analysis)

**Objetivo**: Comparar documentação vs implementação e identificar 5-7 features ausentes.

In [ ]:
def analyze_project_structure():
    """Analisa a estrutura do projeto e retorna overview"""
    
    structure = {
        "backend_routes": [],
        "web_pages": [],
        "types": [],
        "components": []
    }
    
    # Backend routes
    routes_dir = APP_API / "src/routes"
    if routes_dir.exists():
        structure["backend_routes"] = sorted([f.stem for f in routes_dir.glob("*.ts")])
    
    # Web pages
    app_dir = APP_WEB / "src/app"
    if app_dir.exists():
        pages = []
        for root, dirs, files in os.walk(app_dir):
            if "page.tsx" in files or "layout.tsx" in files:
                rel_path = Path(root).relative_to(app_dir)
                pages.append(str(rel_path))
        structure["web_pages"] = sorted(pages)
    
    # Types
    types_dir = PACKAGES / "types/src"
    if types_dir.exists():
        structure["types"] = sorted([f.stem for f in types_dir.glob("*.ts")])
    
    return structure

project = analyze_project_structure()

print("📊 ESTRUTURA DO PROJETO")
print(f"\n🔷 Backend Routes ({len(project['backend_routes'])} endpoints):")
for route in sorted(project['backend_routes']):
    print(f"  • {route}")

print(f"\n🟦 Web Pages ({len(project['web_pages'])} páginas):")
for page in sorted(project['web_pages']):
    if page != ".":
        print(f"  • {page}")

print(f"\n📝 Types ({len(project['types'])} tipos):")
for typ in sorted(project['types']):
    print(f"  • {typ}")

## Fase 1: Análise da Estrutura do Projeto

**Objetivo**: Mapear estrutura, entender propósito, identificar implementação existente.

In [ ]:
import os
import json
import re
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup
REPO_ROOT = Path("/home/crialabs/Documentos/git/saass01")
APP_API = REPO_ROOT / "apps/api"
APP_WEB = REPO_ROOT / "apps/web"
PACKAGES = REPO_ROOT / "packages"

print(f"✅ Repositório carregado: {REPO_ROOT}")
print(f"✅ API: {APP_API}")
print(f"✅ Web: {APP_WEB}")
print(f"✅ Packages: {PACKAGES}")

# Product Manager Workflow: saass01 LMS Feature Identification

**Objetivo**: Implementar um fluxo sistemático para identificar features ausentes, priorizá-las, e criar especificações detalhadas com geração automatizada de issues no GitHub.

**Foco**: saass01 - Monorepo full-stack com 15 endpoints LMS já implementados. Precisa de frontend LMS e features de suporte.

---

## Fases de Execução
1. ✅ **Fase 1**: Análise da Estrutura do Projeto
2. ✅ **Fase 2**: Análise de Lacunas (Gap Analysis)
3. ⏳ **Fase 3**: Priorização com Matriz de Pontuação
4. ⏳ **Fase 4**: Geração de Especificações Detalhadas
5. ⏳ **Fase 5**: Criação de Issues no GitHub
6. ⏳ **Fase 6**: Visualização de Dependências & Roadmap